# 今天我們要來講解下方這個爬取月報的 Function 囉！


# 先來讀資料！


In [21]:
import requests

# 指定爬取月報的網址
url = 'https://mopsov.twse.com.tw/nas/t21/sii/t21sc03_115_5_0.html'
# 抓取網頁
r = requests.get(url)

In [22]:
from io import StringIO
import pandas as pd

# 讓pandas可以讀取中文（測試看看，假如不行讀取中文，就改成 'utf-8'）
r.encoding = 'big5'
# 把剛剛下載下來的網頁的 html 文字檔，利用 StringIO() 包裝成一個檔案給 pandas 讀取
dfs = pd.read_html(StringIO(r.text))

In [38]:
# 將dfs中，row長度介於5~11的table合併（這些才是我們需要的table，其他table不需要）
df = pd.concat([df for df in dfs[2:] if df.shape[1] <= 11 and df.shape[1] > 5])

# 設定column名稱
df.columns = df.columns.get_level_values(1)
df = df.rename(columns={'公司 代號': '公司代號'})

# 在 df['當月營收'] = pd.to_numeric(...) 这一行之前，添加这些调试代码

# 將 df 中的當月營收用 .to_numeric 變成數字，再把其中不能變成數字的部分以 NaN 取代（errors='coerce'）
df['當月營收'] = pd.to_numeric(df['當月營收'], 'coerce')

# 再把當月營收中，出現 NaN 的 row 用 .dropna 整行刪除
df = df[~df['當月營收'].isnull()]

# 刪除「公司代號」中出現「合計」的行數，其中「～」是否定的意思
df = df[df['公司代號'] != '合計']

# # 將「公司代號」與「公司名稱」共同列為 df 的 indexes
df = df.set_index(['公司代號', '公司名稱'])

df.head()

,,當月營收,上月營收,去年當月營收,上月比較 增減(%),去年同月 增減(%),當月累計營收,去年累計營收,前期比較 增減(%),備註
公司代號,公司名稱,,,,,,,,,
1213,大飲,22834,25400,19978,-10.10,14.29,125317,111276,12.61,-
3054,立萬利,58176,86010,6566,-32.36,786.01,280067,28688,876.25,主係因本期產品結構改變致銷售金額增加。
1304,台聚,3720859,4412977,3774657,-15.68,-1.42,18881444,19812980,-4.70,-
1305,華夏,973037,1192324,806203,-18.39,20.69,4839073,4100099,18.02,-
1308,亞聚,521633,512062,472396,1.86,10.42,2335561,2447288,-4.56,-


# 存檔csv (全版本通用)


In [39]:
# ----------- #
# 存取 csv 檔  #
# ----------- #

# 把 df 存成 csv 檔，並且命名為「test.csv」，指定用「utf_8_sig」編碼
df.to_csv('test.csv', encoding='utf_8_sig')

# 讀取名為「test.csv」的 csv 檔，並且指定其中欄位名稱為「公司代號」與「公司名稱」作為 df 的 indexes
df = pd.read_csv('test.csv', index_col=['公司代號', '公司名稱'])
df.head()

,,當月營收,上月營收,去年當月營收,上月比較 增減(%),去年同月 增減(%),當月累計營收,去年累計營收,前期比較 增減(%),備註
公司代號,公司名稱,,,,,,,,,
1213,大飲,22834,25400,19978,-10.10,14.29,125317,111276,12.61,-
3054,立萬利,58176,86010,6566,-32.36,786.01,280067,28688,876.25,主係因本期產品結構改變致銷售金額增加。
1304,台聚,3720859,4412977,3774657,-15.68,-1.42,18881444,19812980,-4.70,-
1305,華夏,973037,1192324,806203,-18.39,20.69,4839073,4100099,18.02,-
1308,亞聚,521633,512062,472396,1.86,10.42,2335561,2447288,-4.56,-


# 存檔sqlite3 (全版本通用)


In [40]:
# --------------- #
# 存取 sqlite3 檔  #
# --------------- #

import sqlite3

# 把 df 存成名為「monthly_report」的 sqlite3 檔，其中 conn 是與 database 的連結
conn = sqlite3.connect('test.sqlite3')
df.to_sql('monthly_report', conn, if_exists='replace')

# 讀取 sqlite3 中名為「monthly_report」的 table，並且指定其中欄位名稱為「公司代號」與「公司代號」作為 df 的 indexes
df = pd.read_sql('select * from monthly_report', conn, index_col=['公司代號', '公司名稱'])
df.head()

,,當月營收,上月營收,去年當月營收,上月比較 增減(%),去年同月 增減(%),當月累計營收,去年累計營收,前期比較 增減(%),備註
公司代號,公司名稱,,,,,,,,,
1213,大飲,22834,25400,19978,-10.10,14.29,125317,111276,12.61,-
3054,立萬利,58176,86010,6566,-32.36,786.01,280067,28688,876.25,主係因本期產品結構改變致銷售金額增加。
1304,台聚,3720859,4412977,3774657,-15.68,-1.42,18881444,19812980,-4.70,-
1305,華夏,973037,1192324,806203,-18.39,20.69,4839073,4100099,18.02,-
1308,亞聚,521633,512062,472396,1.86,10.42,2335561,2447288,-4.56,-
